# Environment Validation

Development notebook for validating the Netflix data engineering environment before components are automated with Airflow.

### Development approach

- **Notebooks** → exploration and debugging
- **`src/`** → reusable Python logic
- **`sql/`** → transformations and data-quality queries
- **`dags/`** → Airflow orchestration

In [1]:
# ------------------------------------------------------------
# Basic Python environment check
# ------------------------------------------------------------
# Before testing databases or pipelines, we first confirm that
# this notebook is executing Python correctly.

import sys
import platform

print("Python version:", sys.version.split()[0])
print("Operating system:", platform.system())
print("Machine architecture:", platform.machine())

Python version: 3.10.11
Operating system: Darwin
Machine architecture: x86_64


In [2]:
# ------------------------------------------------------------
# Project environment confirmation
# ------------------------------------------------------------
# Confirms that the notebook is running inside the expected
# Netflix data engineering development environment.

from pathlib import Path
import sys

PROJECT_NAME = "netflix-data-engineering"

print(f"Project: {PROJECT_NAME}")
print(f"Python executable: {sys.executable}")
print(f"Working directory: {Path.cwd()}")
print("Development environment: READY")

Project: netflix-data-engineering
Python executable: /Users/mac/Documents/netflix-data-engineering/.venv/bin/python
Working directory: /Users/mac/Documents/netflix-data-engineering/notebooks
Development environment: READY


## PostgreSQL Connectivity

Validate that the local development environment can connect to the existing Netflix PostgreSQL database before the connection logic is moved into Airflow.

In [ ]:
# Import os so Python can read environment variables.
import os

# Import Path so we can locate the project's .env file reliably.
from pathlib import Path

# Import Psycopg, the PostgreSQL driver used by the development environment.
import psycopg

# Import load_dotenv so secrets can be loaded from .env instead of hard-coded.
from dotenv import load_dotenv


# Get the current notebook working directory.
current_path = Path.cwd()

# Move to the project root when the notebook is running from /notebooks.
project_root = current_path.parent if current_path.name == "notebooks" else current_path

env_path = project_root / ".env"

load_dotenv(env_path)

# Read the PostgreSQL configuration from environment variables.
db_host = os.getenv("NETFLIX_DB_HOST")
db_port = os.getenv("NETFLIX_DB_PORT")
db_name = os.getenv("NETFLIX_DB_NAME")
db_user = os.getenv("NETFLIX_DB_USER")
db_password = os.getenv("NETFLIX_DB_PASSWORD")

# Confirm that the required configuration was loaded without printing the password.
print("Database host:", db_host)
print("Database port:", db_port)
print("Database name:", db_name)
print("Database user:", db_user)
print("Password loaded:", bool(db_password))

Database host: 172.20.10.2
Database port: 5432
Database name: postgres
Database user: powerbi_reader
Password loaded: True


## Test the actual PostgreSQL connection

In [5]:
# Build a PostgreSQL connection using the values loaded from .env.
# The timeout prevents the notebook from hanging for a long time if the server is unreachable.
with psycopg.connect(
    host=db_host,
    port=db_port,
    dbname=db_name,
    user=db_user,
    password=db_password,
    connect_timeout=5,
) as connection:

    # Open a cursor so we can execute SQL against PostgreSQL.
    with connection.cursor() as cursor:

        # Ask PostgreSQL for basic server information.
        cursor.execute(
            """
            SELECT
                current_database(),
                current_user,
                version();
            """
        )

        # Retrieve the single row returned by PostgreSQL.
        database_name, database_user, postgres_version = cursor.fetchone()

        # Display safe connection information.
        print("Connection status: SUCCESS")
        print("Connected database:", database_name)
        print("Connected user:", database_user)
        print("PostgreSQL version:", postgres_version.split(",")[0])

Connection status: SUCCESS
Connected database: postgres
Connected user: powerbi_reader
PostgreSQL version: PostgreSQL 18.3 on x86_64-apple-darwin24.6.0


## Database Structure Inspection

Inspect the existing PostgreSQL schemas and tables before building the automated staging and transformation pipeline.

The purpose is to understand the current database structure, identify source tables, and determine how the existing SQL should be mapped into the Airflow workflow.

In [ ]:
# Connect to the existing PostgreSQL database.
with psycopg.connect(
    host=db_host,
    port=db_port,
    dbname=db_name,
    user=db_user,
    password=db_password,
) as connection:

    with connection.cursor() as cursor:

        cursor.execute(
            """
            SELECT schema_name
            FROM information_schema.schemata
            WHERE schema_name NOT LIKE 'pg_%'
              AND schema_name <> 'information_schema'
            ORDER BY schema_name;
            """
        )

        # Fetch every schema returned by PostgreSQL.
        schemas = cursor.fetchall()

        # Display the schemas found in the database.
        print("Available schemas:")

        # Loop through each schema result.
        for schema in schemas:

            # Print only the schema name from the returned tuple.
            print("-", schema[0])

Available schemas:
- analytics
- public


## Inspect all tables

In [ ]:
with psycopg.connect(
    host=db_host,
    port=db_port,
    dbname=db_name,
    user=db_user,
    password=db_password,
) as connection:

    with connection.cursor() as cursor:

        # Query every user-accessible base table.
        cursor.execute(
            """
            SELECT
                table_schema,
                table_name
            FROM information_schema.tables
            WHERE table_type = 'BASE TABLE'
              AND table_schema NOT LIKE 'pg_%'
              AND table_schema <> 'information_schema'
            ORDER BY table_schema, table_name;
            """
        )

        # Retrieve all table records.
        tables = cursor.fetchall()

        print("Available tables:")

        # Loop through the tables discovered.
        for schema_name, table_name in tables:

            # Print the fully-qualified table name.
            print(f"- {schema_name}.{table_name}")

Available tables:
- uat_olap.bridge_title_country
- uat_olap.bridge_title_genre
- uat_olap.dim_country
- uat_olap.dim_genre
- uat_olap.dim_person
- uat_olap.dim_title
- uat_olap.fact_credits
- uat_olap.fact_title_metrics


## Get table row counts

In [ ]:
# Import SQL composition helpers so table names can be inserted safely.
from psycopg import sql


# Connect to PostgreSQL for row-count profiling.
with psycopg.connect(
    host=db_host,
    port=db_port,
    dbname=db_name,
    user=db_user,
    password=db_password,
) as connection:

    with connection.cursor() as cursor:

        cursor.execute(
            """
            SELECT
                table_schema,
                table_name
            FROM information_schema.tables
            WHERE table_type = 'BASE TABLE'
              AND table_schema NOT LIKE 'pg_%'
              AND table_schema <> 'information_schema'
            ORDER BY table_schema, table_name;
            """
        )

        # Store the tables returned by PostgreSQL.
        tables = cursor.fetchall()

        for schema_name, table_name in tables:

            # Build a safe COUNT query using PostgreSQL identifiers.
            count_query = sql.SQL(
                "SELECT COUNT(*) FROM {}.{}"

            # sql.Identifier() here is that schema/table names are identifiers, not normal query values
            ).format(
                sql.Identifier(schema_name),
                sql.Identifier(table_name),
            )

            # Execute the row-count query.
            cursor.execute(count_query)

            # Retrieve the row count.
            row_count = cursor.fetchone()[0]

            # Display the fully-qualified table name and count.
            print(
                f"{schema_name}.{table_name}: "
                f"{row_count:,} rows"
            )

uat_olap.bridge_title_country: 6,528 rows
uat_olap.bridge_title_genre: 15,088 rows
uat_olap.dim_country: 109 rows
uat_olap.dim_genre: 19 rows
uat_olap.dim_person: 54,589 rows
uat_olap.dim_title: 5,849 rows
uat_olap.fact_credits: 77,800 rows
uat_olap.fact_title_metrics: 5,849 rows


## Inspect table columns

In [ ]:
with psycopg.connect(
    host=db_host,
    port=db_port,
    dbname=db_name,
    user=db_user,
    password=db_password,
) as connection:

    with connection.cursor() as cursor:

        # Read column definitions for all user-created tables.
        cursor.execute(
            """
            SELECT
                table_schema,
                table_name,
                ordinal_position,
                column_name,
                data_type,
                is_nullable
            FROM information_schema.columns
            WHERE table_schema NOT LIKE 'pg_%'
              AND table_schema <> 'information_schema'
            ORDER BY
                table_schema,
                table_name,
                ordinal_position;
            """
        )

        # Retrieve all column definitions.
        columns = cursor.fetchall()

        # Keep track of the table currently being printed.
        current_table = None

        # Loop through the metadata.
        for (
            schema_name,
            table_name,
            position,
            column_name,
            data_type,
            nullable,
        ) in columns:

            # Build the fully-qualified table name.
            full_table_name = f"{schema_name}.{table_name}"

            # Print a new heading whenever the table changes.
            if full_table_name != current_table:

                # Display the table name.
                print(f"\n{full_table_name}")

                # Update the current table tracker.
                current_table = full_table_name

            print(
                f"  {position}. "
                f"{column_name} | "
                f"{data_type} | "
                f"nullable={nullable}"
            )


uat_olap.bridge_title_country
  1. title_sk | bigint | nullable=NO
  2. country_sk | bigint | nullable=NO

uat_olap.bridge_title_genre
  1. title_sk | integer | nullable=NO
  2. genre_sk | integer | nullable=NO

uat_olap.dim_country
  1. country_sk | bigint | nullable=NO
  2. country_id | integer | nullable=NO
  3. name | character varying | nullable=NO

uat_olap.dim_genre
  1. genre_sk | integer | nullable=NO
  2. genre_id | integer | nullable=NO
  3. name | character varying | nullable=NO

uat_olap.dim_person
  1. person_sk | integer | nullable=NO
  2. person_id | integer | nullable=NO
  3. name | character varying | nullable=NO

uat_olap.dim_title
  1. title_sk | integer | nullable=NO
  2. title_id | character varying | nullable=NO
  3. title_name | character varying | nullable=YES
  4. type | character varying | nullable=NO
  5. age_certification | character varying | nullable=YES
  6. release_year | integer | nullable=YES

uat_olap.fact_credits
  1. title_sk | integer | nullable=